# Module 1: Foundations

Build a minimal Strands agent, run it, and inspect every step of the **agentic loop** in action.

![Strands agent loop: Input and Context flows into Reasoning LLM, then Tool Selection, then Tool Execution, which loops back to Reasoning until done, then Response](./agent-loop.png)

The agentic loop is the core of every Strands agent:

1. **Input & Context**: the user prompt enters the loop
2. **Reasoning (LLM)**: the model decides what to do next
3. **Tool Selection**: if it needs data, the model picks a tool
4. **Tool Execution**: your Python function runs, feeding its result back into step 2
5. **Response**: the loop exits once the model has enough information

You write the tools. Strands runs the loop.

**Prerequisites:** Python 3.10+, AWS credentials with Amazon Bedrock access.

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


---

## AI Agent Primitives

An agent = a model given tools, instructions, and memory, running in a loop until the goal is met.

| Primitive | What it is | In Strands |
|-----------|-----------|-----------|
| **Model (LLM)** | The reasoning engine: plans, decides, responds | `BedrockModel(model_id=...)` passed to `Agent` |
| **Tools** | Python functions the model can call to act | `@tool` decorated functions |
| **Prompt & Instructions** | Role, goals, and guardrails that steer behavior | `system_prompt=` parameter on `Agent` |
| **Memory & Context** | Conversation history and state across turns | `agent.messages` list, grows each turn |

These four primitives are the building blocks. Every multi-agent pattern in this workshop is built from combinations of them.

![AI Agent Primitives: AGENT in center connected to Model (LLM), Tools, Prompt & Instructions, Memory & Context](./primitives.png)

---

## Part 1: Define a Tool

Tools are plain Python functions decorated with `@tool`. The model reads the **docstring** to decide when and how to call them. The docstring is the routing logic, not code.

Write docstrings for the model, not for other developers.

In [2]:
from strands import Agent, tool

@tool
def get_market_data(segment: str) -> str:
    """Get subscription market data for a customer segment.

    Args:
        segment: The customer segment to look up (e.g. 'premium', 'standard', 'enterprise')
    """
    data = {
        "premium": {"willingness_to_pay": "68%", "avg_monthly": "$22", "churn_rate": "8%"},
        "standard": {"willingness_to_pay": "41%", "avg_monthly": "$14", "churn_rate": "18%"},
        "enterprise": {"willingness_to_pay": "85%", "avg_monthly": "$45", "churn_rate": "4%"},
    }
    row = data.get(segment.lower(), {"error": f"no data for segment '{segment}'"})
    return str(row)

/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/pydantic/plugin/_schema_validator.py:39: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  plugins = get_plugins()


Tool defined. Docstring the model will read:
Get subscription market data for a customer segment.

    Args:
        segment: The customer segment to look up (e.g. 'premium', 'standard', 'enterprise')
    


---

## Part 2: Create and Run the Agent

Wire the tool into an `Agent` with a system prompt and call it.

In [3]:
agent = Agent(
    tools=[get_market_data],
    system_prompt="You are a market research assistant. Use your tools to answer questions accurately.",
)

result = agent("What percentage of premium segment customers are willing to pay for a subscription tier?")


Tool #1: get_market_data


Based on the market data retrieved, **68% of premium segment customers** are willing to pay for a subscription tier. Here

 are some additional insights from the data:

- 

💰 **Average Monthly Spend:** $22 

per customer
- 📉 **Churn Rate:** 

8%

This suggests that the premium segment has

 a strong willingness to pay, with over two-thirds of customers open to a

 subscription model. The relatively low churn rate of

 8% also indicates good retention

 within this segment.

---

## Part 3: Inspect the Agent Loop

Every step is stored in `agent.messages`. Let's see what happened inside the loop.

In [ ]:
import json as _json

for i, msg in enumerate(agent.messages):
    role = msg["role"]
    content = msg.get("content", [])

    if role == "user":
        if isinstance(content, str):
        elif isinstance(content, list):
            for block in content:
                if not isinstance(block, dict):
                    continue
                if "text" in block and block["text"]:
                elif "toolResult" in block:
                    # Tool results arrive as user-role messages with toolResult blocks
                    tr = block["toolResult"]
                    for item in tr.get("content", []):
                        if "text" in item:

    elif role == "assistant":
        blocks = content if isinstance(content, list) else [{"text": str(content)}]
        for block in blocks:
            if "text" in block and block["text"]:
            elif "toolUse" in block:
                tu = block["toolUse"]

---

## Part 4: Metrics

Every `Agent.__call__()` returns an `AgentResult`. The `result.metrics` attribute gives you token usage and loop stats with no extra setup.

This is the built-in observability you use in notebooks. For production traces in CloudWatch, see the `production/` folder of each module.

In [ ]:
# Lens 1: metrics on every AgentResult
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})

---

## Part 5: Multi-turn Conversation

The agent keeps its conversation history across calls. Each `agent(...)` call adds to the same context window. The agent remembers what happened in previous turns.

In [6]:
# Multi-turn: the agent remembers previous turns
agent3 = Agent(
    tools=[get_market_data],
    system_prompt="You are a market research assistant. Use your tools to answer questions accurately.",
)

agent3("What is the willingness to pay for the premium segment?")
agent3("And for the standard segment?")
response = agent3("Which segment has the lower churn rate between those two?")


Tool #1: get_market_data


Based on the market data, the **willingness to pay for the premium segment is 68%**. Here

 are some additional details for context:

- 

💰 **Average Monthly Spend:** $22
- 📉

 **Churn Rate:** 8%

This indicates that a strong

...

 lower churn rate at **8%**, compared

 to the standard segment's **18%**. That's

 a significant difference of 10 percentage points, indicating that premium customers are considerably more loyal

 and less likely to cancel their subsc

riptions.
Conversation length: 10 messages


---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` | Python function the model can call; docstring is the routing logic |
| `Agent(tools=[], system_prompt=...)` | Assembles model + tools + instructions |
| `agent.messages` | Full loop history: user turns, LLM responses, tool calls, tool results |
| `result.metrics.get_summary()` | Token usage and cycle count on every call |
| Multi-turn | Each `agent(...)` call extends the same context window |

---

## What is next

In **Module 2: Single Agent**, you will use these same building blocks with three real business intelligence tools to build a complete Decision Intelligence agent for the NovaCart brief.